In [ ]:
import os
import re
import json
import time

from pathlib import Path
from collections import defaultdict
from semanticher.data import load_table_list, load_tables

from langchain_openai.chat_models.base import BaseChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# =========================
# Base folders
# =========================
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_msv_base"
CANDIDATE_DIR   = BASE_RESULT_DIR / "Result_msv_base" / "Candidate_msv_base"
LOG_DIR         = BASE_RESULT_DIR / "Promptinput_msv_base"
SUMMARY_DIR     = BASE_RESULT_DIR / "Tokenusage_Duration_msv_base"

os.makedirs(CANDIDATE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,       exist_ok=True)
os.makedirs(SUMMARY_DIR,   exist_ok=True)


# =========================
# Data / ontology
# =========================
ONTOLOGY_FILE = REPO_ROOT / "data" / "ontology" / "BED.owl"

with open(ONTOLOGY_FILE, "r", encoding="utf-8") as f:
    raw_ontology_text = f.read()

lines = raw_ontology_text.splitlines(keepends=True)

ontology_text = "".join(
    line for line in lines
    if any(key in line for key in [
        "<owl:Class",
        "</owl:Class>",
        "rdfs:subClassOf",
        "owl:equivalentClass",
    ])
)

print("Ontology file:", ONTOLOGY_FILE)
print("Raw ontology length (chars):", len(raw_ontology_text))
print("Compressed ontology length (chars):", len(ontology_text))

df_table_list = load_table_list()
dict_tables   = load_tables()


def dataframe_to_markdown(df):
    subset  = df
    headers = "| " + " | ".join(subset.columns) + " |"
    sep     = "| " + " | ".join(["---"] * len(subset.columns)) + " |"

    rows = []
    for _, row in subset.iterrows():
        row_str = "| " + " | ".join(map(str, row.values)) + " |"
        rows.append(row_str)

    return headers + "\n" + sep + "\n" + "\n".join(rows)


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_candidate_path(model_names):
    tag = build_experiment_tag(model_names)
    return CANDIDATE_DIR / f"candidate_msv_base_{tag}.json"


def build_logs_path(model_names):
    tag = build_experiment_tag(model_names)
    return LOG_DIR / f"msv_base_{tag}_logs.json"


def build_summary_path(model_names):
    tag = build_experiment_tag(model_names)
    return SUMMARY_DIR / f"summary_msv_base_{tag}.json"

In [ ]:
# =========================
# LLMs
# =========================
llm1 = BaseChatOpenAI(
    model="deepseek-chat",
    openai_api_key="",
    openai_api_base="https://api.deepseek.com",
    temperature=0,
    max_tokens=8192,
)

llm2 = BaseChatOpenAI(
    model="gemini-2.0-flash",
    openai_api_key="",
    openai_api_base="https://generativelanguage.googleapis.com/v1beta/openai",
    temperature=0,
    max_tokens=None,
)

llm3 = BaseChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key="",
    openai_api_base="https://api.openai.com/v1",
    temperature=0,
    max_tokens=None,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a knowledgeable assistant who helps map tabular data columns to ontology classes.
            You have expert knowledge in semantic table annotation, ontology reasoning, RDF/OWL class hierarchies, and URI-level ontology prediction.
            Your task is to annotate all columns of a given table using only the complete ontology file provided in OWL format. No preprocessed ontology structure is provided.
            You must infer hierarchy, equivalent classes, and valid prediction paths by yourself from the ontology.
            """
        ),
        (
            "human",
            """
            We have a table named "{table_name}" which has several columns. Below is the table in Markdown format, including column headers and the complete table content:

            {table_in_markdown}

            Below is the complete ontology file in OWL format:

            {ontology_text}

            Task:
            For each column, output multiple plausible ontology class paths (Top-k).

            Ontology reasoning rules:
            - You must rely only on the ontology file above.
            - The ontology defines class hierarchy through constructs such as rdfs:subClassOf.
            - You must infer valid Level1 and Level2 classes by yourself from the ontology.
            - Level1 class: a class that represents a top-level concept in the ontology hierarchy.
            - Level2 class: a class whose direct parent is a Level1 class.
            - If a class is deeper than Level2, do NOT output that deeper class directly. Move upward and output only the corresponding Level1 or Level1 -> Level2 path.

            Equivalent-class rule:
            - If multiple classes are semantically equivalent, for example through owl:equivalentClass, you may treat them as the same concept.
            - In such cases, choose one URI by yourself and output that URI.
            - Be consistent: for the same concept, avoid switching between equivalent URIs unnecessarily.

            Annotation rules:
            - Analyze all columns jointly.
            - Use column names, sample values, data types, units, and table context as evidence.
            - Prefer the most specific valid class supported by the evidence, but the final output must still be restricted to Level1 or Level1 -> Level2.
            - Output URIs only. Do NOT output labels, comments, nicknames, or invented identifiers.
            - Every predicted URI must appear in the ontology file and must correspond to a class.
            - If no suitable class exists, return an empty paths list.

            Output format:
            - Output ONLY a valid JSON array.
            - Do NOT include any explanatory text.
            - Each item in the array must correspond to one input column, in the same order as the table.
            - Each item must have the following structure:

            {{
                "column_name": "<exact column header>",
                "paths": [
                    ["<Level1_URI>", confidence],
                    ["<Level1_URI>", "<Level2_URI>", confidence]
                ]
            }}

            Path rules:
            - Each element in "paths" represents exactly one class path.
            - If a path contains only a Level1 class, use:
              ["<Level1_URI>", confidence]
            - If a path contains both Level1 and Level2 classes, use:
              ["<Level1_URI>", "<Level2_URI>", confidence]
            - Confidence is the confidence of the whole path, not separate nodes.
            - Confidence values must be numbers between 0 and 1.
            - Order paths by descending confidence.
            - If no suitable class exists for a column, return:
              {{"column_name": "<exact column header>", "paths": []}}

            Important:
            - You must infer the hierarchy from the ontology file itself.
            - You must output URIs only, not class labels.
            - You must output only Level1 or Level1 -> Level2 classes.
            - Do NOT output classes deeper than Level2.
            - If equivalent classes exist, choose one URI yourself and use it consistently.
            - Do NOT invent URIs.
            - Before producing the final answer, verify that every predicted URI:
              1) exists in the ontology,
              2) corresponds to a class,
              3) satisfies the Level1 or Level2 requirement.
            - Return ONLY the JSON array.
            """
        ),
    ]
)

CHAIN_REGISTRY = {
    "deepseek-chat":    prompt | llm1,
    "gemini-2.0-flash": prompt | llm2,
    "gpt-4.1-mini":     prompt | llm3,
}

In [ ]:
# =========================
# LLM output processing
# =========================
def extract_json_from_response(text):
    """
    Robust JSON extractor from LLM responses.
    """
    m = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    s = m.group(1).strip() if m else text.strip()

    if '\\"' in s:
        try:
            unescaped = json.loads(f'"{s}"')
            if isinstance(unescaped, str):
                s = unescaped.strip()
        except Exception:
            pass

    start = s.find("[")
    end   = s.rfind("]")
    if start != -1 and end != -1 and end > start:
        return s[start:end + 1]

    start = s.find("{")
    end   = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        return s[start:end + 1]

    return s


def normalize_confidence(x):
    try:
        x = float(x)
    except Exception:
        return None
    if x < 0:
        x = 0.0
    if x > 1:
        x = 1.0
    return round(x, 4)


def process_llm_output(llm_output_str, table_id, table_name, table_columns):
    """
    Convert raw LLM JSON output to per-column path-level records.
    LLM outputs URIs; stored as-is in "class" field for later uri_to_name conversion.

    Output record format:
    {
        "table_id": ...,
        "table_name": ...,
        "column_name": ...,
        "column_id": ...,
        "class": [
            ["<Level1_URI>", conf],
            ["<Level1_URI>", "<Level2_URI>", conf]
        ]
    }
    """
    results = []

    try:
        parsed = json.loads(llm_output_str)
        if not isinstance(parsed, list):
            parsed = []
    except json.JSONDecodeError as e:
        print(f"JSON fail on {table_id}: {e}")
        parsed = []

    parsed_by_index = {}
    for i, col in enumerate(parsed):
        if isinstance(col, dict):
            parsed_by_index[i] = col

    for i, column_name in enumerate(table_columns):
        col       = parsed_by_index.get(i, {})
        raw_paths = col.get("paths", [])

        best_scores = {}

        for p in raw_paths:
            if not isinstance(p, list):
                continue

            # Level 1 only
            if len(p) == 2:
                level1, conf = p
                if not isinstance(level1, str) or not level1:
                    continue
                conf = normalize_confidence(conf)
                if conf is None:
                    continue
                key = (level1,)
                best_scores[key] = max(best_scores.get(key, 0), conf)

            # Level 1 -> Level 2
            elif len(p) == 3:
                level1, level2, conf = p
                if not isinstance(level1, str) or not level1:
                    continue
                if not isinstance(level2, str) or not level2:
                    continue
                conf = normalize_confidence(conf)
                if conf is None:
                    continue
                key = (level1, level2)
                best_scores[key] = max(best_scores.get(key, 0), conf)

        normalized_paths = []
        for key, conf in best_scores.items():
            if len(key) == 1:
                normalized_paths.append([key[0], conf])
            else:
                normalized_paths.append([key[0], key[1], conf])

        normalized_paths.sort(key=lambda x: x[-1], reverse=True)

        results.append({
            "table_id":    table_id,
            "table_name":  table_name,
            "column_name": column_name,
            "column_id":   i,
            "class":       normalized_paths,  # URI，eval 时再转 name
        })

    return results

In [ ]:
# =========================
# Single experiment runner
# =========================
def run_single_experiment(model_names):
    chains     = [CHAIN_REGISTRY[name] for name in model_names]
    num_agents = len(model_names)

    candidate_file_path = build_candidate_path(model_names)
    logs_path           = build_logs_path(model_names)
    summary_path        = build_summary_path(model_names)

    print("=" * 80)
    print(f"Running experiment: {build_experiment_tag(model_names)}")
    print("Models:            ", model_names)
    print("Candidate file:    ", candidate_file_path)
    print("Logs file:         ", logs_path)
    print("=" * 80)

    start_time = time.perf_counter()

    all_llm_records    = []
    candidate_results  = []

    for table_id in df_table_list["table_id"].unique():
        table             = dict_tables[table_id]
        table_name        = df_table_list[df_table_list["table_id"] == table_id]["table_name"].values[0]
        table_in_markdown = dataframe_to_markdown(table)
        table_columns     = list(table.columns)

        print("+" * 70)
        print(table_id, table_name)

        input_dict = {
            "table_name":       table_name,
            "table_in_markdown": table_in_markdown,
            "ontology_text":    ontology_text,
        }

        for agent_id in range(num_agents):
            model_name      = model_names[agent_id]
            chain           = chains[agent_id]

            prompt_template  = chain.first
            formatted_prompt = prompt_template.format(**input_dict)

            if isinstance(formatted_prompt, str):
                prompt_text = formatted_prompt.strip()
            else:
                messages    = formatted_prompt.to_messages()
                prompt_text = "\n".join([f"[{m.type.upper()}]\n{m.content}" for m in messages])

            response    = chain.invoke(input_dict)
            output_text = extract_json_from_response(response.content)

            usage_raw   = response.response_metadata.get("token_usage", {})
            usage_clean = {
                "input_tokens":  usage_raw.get("prompt_tokens", 0),
                "output_tokens": usage_raw.get("completion_tokens", 0),
                "total_tokens":  usage_raw.get("total_tokens", 0),
            }

            parsed_results = process_llm_output(
                llm_output_str=output_text,
                table_id=table_id,
                table_name=table_name,
                table_columns=table_columns,
            )

            for item in parsed_results:
                candidate_results.append({
                    "table_id":    item["table_id"],
                    "table_name":  item["table_name"],
                    "model":       model_name,
                    "column_name": item["column_name"],
                    "column_id":   item["column_id"],
                    "class":       item["class"],   # URI，eval 时再转 name
                })

            all_llm_records.append({
                "table_id":   table_id,
                "table_name": table_name,
                "model":      model_name,
                "prompt":     prompt_text,
                "output":     response.content,
                "usage":      usage_clean,
            })

            print(f"  [{model_name}] done")

    end_time        = time.perf_counter()
    elapsed_time    = end_time - start_time
    elapsed_minutes = elapsed_time / 60

    token_summary = {
        "total":    {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "by_model": defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}),
    }

    for rec in all_llm_records:
        model = rec.get("model", "unknown_model")
        usage = rec.get("usage", {})
        for key in ["input_tokens", "output_tokens", "total_tokens"]:
            value = usage.get(key, 0) or 0
            token_summary["total"][key] += value
            token_summary["by_model"][model][key] += value

    summary = {
        "experiment": build_experiment_tag(model_names),
        "models":     model_names,
        "run_time": {
            "seconds": round(elapsed_time, 2),
            "minutes": round(elapsed_minutes, 2),
        },
        "token_usage_summary": {
            "total":    token_summary["total"],
            "by_model": dict(token_summary["by_model"]),
        },
    }

    with open(candidate_file_path, "w", encoding="utf-8") as f:
        json.dump(candidate_results, f, ensure_ascii=False, indent=2)

    with open(logs_path, "w", encoding="utf-8") as f:
        json.dump(all_llm_records, f, ensure_ascii=False, indent=2)

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"Saved candidate -> {candidate_file_path}")
    print(f"Saved logs      -> {logs_path}")
    print(f"Saved summary   -> {summary_path}")

In [ ]:
def run_all_experiments():
    for model_names in EXPERIMENTS:
        run_single_experiment(model_names=model_names)

run_all_experiments()